# FN-PLEFP8-UPLOAD — FP8 PLE overlay for Qwen3.8-Flash-Next W4A16 (HF → private Kaggle model)

Utility notebook (CPU, **internet ON**, no GPU, no competition data — it is never submitted).

* Source: `albucino/Qwen3.8-Flash-Next-W4A16-FP8PLE` at commit `c6af9fee7ad2cb8ef44b534a99b8c1b81fb0f6da`.
* Uploads only the 15 files that differ from Intel's checkpoint (`4c67bf6`): 10 FP8 PLE
  shards (official RadixArk FP8 table + scale) + `config.json` (adds `ple_embedding_dtype: float8_e4m3fn`) +
  `model.safetensors.index.json` + LICENSE / README / SHA256SUMS. ≈51.2 GB.
* The 15 main W4A16 shards are byte-identical to Intel's (SHA256 checked) and are taken at run time from
  `dfranzen/intel-qwen3.8-flash-next-w4a16-autoround`.
* Target: private model `foysalemonshanto/qwen38-flash-next-fp8ple-overlay` (framework Transformers).

**Before running:** Add-ons → Secrets → attach either `KAGGLE_API_TOKEN`, or `KAGGLE_USERNAME` + `KAGGLE_KEY`, of
**foysalemonshanto**. The notebook never prints them, and it checks the account name before downloading anything.
Every file is SHA256-verified against the pinned Hugging Face commit before upload.


In [ ]:
# ==================================================================== CELL 1 — config + guards
import os, sys, json, time, shutil, hashlib, subprocess, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

class Stop(BaseException):
    """not an Exception: never swallowed by `except Exception`, and it halts Run All (SystemExit would not)"""

T0 = time.time()
if os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}:
    raise Stop("utility notebook - nothing to do in a competition rerun")
EXPECTED_USER = "foysalemonshanto"
MODEL_SLUG = "qwen38-flash-next-fp8ple-overlay"
REPO = "albucino/Qwen3.8-Flash-Next-W4A16-FP8PLE"
REVISION = "c6af9fee7ad2cb8ef44b534a99b8c1b81fb0f6da"
OVERLAY = json.loads(r"""{
 "model-plefp8-00000.safetensors": {
  "size": 5200027199,
  "sha256": "dc2e845b7edd35bda92834fba3626bf7d199e28d6aceac99fee654aade390cfd"
 },
 "model-plefp8-00001.safetensors": {
  "size": 5200027202,
  "sha256": "899eaa0716e28594468a1389ee58cb065c23907f1270de3831f5ecb0a4f82d56"
 },
 "model-plefp8-00002.safetensors": {
  "size": 5200027198,
  "sha256": "06fd8a11abf0419a669f89397b8d70dd6ff42d401e6b2a037c65e49704faaf71"
 },
 "model-plefp8-00003.safetensors": {
  "size": 5200027189,
  "sha256": "c6aaa1fc08e84eced3c8151ac8679ed943888eba0fcef2556963693430f95bd9"
 },
 "model-plefp8-00004.safetensors": {
  "size": 5200027190,
  "sha256": "d94e97c96d3ea09208614da016960f3f4b429f47a044c898a51b493f42f74ba2"
 },
 "model-plefp8-00005.safetensors": {
  "size": 5200027190,
  "sha256": "03d5d4792e14a4ab55bae50bb459624b01786f818bbcfcaed1a5d0235af484c6"
 },
 "model-plefp8-00006.safetensors": {
  "size": 5200027190,
  "sha256": "586cccbc12383021bc9bc02f206d9b19fbd1672373ed9a5d91e3b0ce34c2418f"
 },
 "model-plefp8-00007.safetensors": {
  "size": 5200027190,
  "sha256": "c4a23bcc10f3cde6b633e82c282a9a518e3a64d433628283e1bc592c94cf3d6c"
 },
 "model-plefp8-00008.safetensors": {
  "size": 5200027190,
  "sha256": "2dc8098c0d020bff277c9cf499a6b908e17836b70a5f949dfa24793371c9a87e"
 },
 "model-plefp8-00009.safetensors": {
  "size": 4400023163,
  "sha256": "61de98b89bb79f386795787d7a76827a26f1e292c26edbb0a1b613da455f5a9c"
 },
 "config.json": {
  "size": 266471,
  "sha256": "2925a0d0989384e0ef7973a14c97ba9c8dfc82c4ca3684d2e83cf4f57db6c172"
 },
 "model.safetensors.index.json": {
  "size": 23657880,
  "sha256": "eb08a649acf89aa2d0c5c1858704cba8e86cdf2b93dec8cf9a89024292a3cc5a"
 },
 "LICENSE": {
  "size": 3235,
  "sha256": "a0dc422560841fd68e06d974907f8b4c709bca44a67daad2b528437bdf676c08"
 },
 "README.md": {
  "size": 15547,
  "sha256": "3c1ecf16c50cdca78d208dedc5a70cc974e154a83e2e5ea62f6df072cdd37497"
 },
 "SHA256SUMS": {
  "size": 9453,
  "sha256": "e8a0a6ebd68053b35d28f08e6164354351867c9c9fcf9ada81fee5a557d61b80"
 }
}""")
TOTAL = sum(v["size"] for v in OVERLAY.values())
LICENSE_NAME = "Other"          # Qwen Community License 1.0 (LICENSE file is uploaded with the model)
NOTES = f"FP8 PLE overlay of {REPO}@{REVISION[:7]} for Intel/Qwen3.8-Flash-Next-W4A16-AutoRound@4c67bf6"
OUT = Path("/kaggle/working"); OUT.mkdir(parents=True, exist_ok=True)
REPORT = {"repo": REPO, "revision": REVISION, "model": f"{EXPECTED_USER}/{MODEL_SLUG}", "steps": {}}

def step(name, ok, **info):
    REPORT["steps"][name] = {"ok": bool(ok), "t_sec": round(time.time() - T0, 1), **info}
    (OUT / "upload_report.json").write_text(json.dumps(REPORT, indent=2, default=str))
    print(("PASS " if ok else "FAIL ") + name, json.dumps(info, default=str)[:1500], flush=True)
    if not ok:
        raise Stop(f"stopped at {name}")

print(f"{len(OVERLAY)} files, {TOTAL / 1e9:.2f} GB from {REPO}@{REVISION[:7]}")


In [ ]:
# ==================================================================== CELL 2 — credentials (Kaggle Secrets) + account check
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kagglehub>=1.0.2"], check=False)
loaded = []
try:
    from kaggle_secrets import UserSecretsClient
    sc = UserSecretsClient()
    for names in (["KAGGLE_API_TOKEN"], ["KAGGLE_USERNAME", "KAGGLE_KEY"]):
        try:
            vals = [sc.get_secret(n) for n in names]
        except Exception:
            continue
        if all(vals):
            for n, v in zip(names, vals):
                os.environ[n] = v.strip()          # value is never printed
            loaded = names
            break
except Exception as e:
    print("kaggle_secrets unavailable:", type(e).__name__)
import kagglehub
try:
    who = kagglehub.whoami(verbose=False).get("username")
except Exception as e:
    who = None
    print("whoami failed:", type(e).__name__)
step("auth", who == EXPECTED_USER, secrets_used=loaded, username=who, expected=EXPECTED_USER, kagglehub=kagglehub.__version__)


In [ ]:
# ==================================================================== CELL 3 — staging disk
def free_gb(p):
    try:
        Path(p).mkdir(parents=True, exist_ok=True)
        return shutil.disk_usage(p).free / 1e9
    except Exception:
        return 0.0

cands = {p: round(free_gb(p), 1) for p in ("/kaggle/tmp", "/tmp")}
STAGE_ROOT = max(cands, key=cands.get)
NEED_ALL = TOTAL / 1e9 + 4
# one instance if everything fits at once; otherwise two parts uploaded one after the other (the smoke notebook
# merges every instance of the model, so both layouts load the same way)
names = sorted(OVERLAY)
shards = [n for n in names if n.endswith(".safetensors")]
small = [n for n in names if not n.endswith(".safetensors")]
if cands[STAGE_ROOT] >= NEED_ALL:
    PARTS = {"default": names}
else:
    half = len(shards) // 2
    PARTS = {"part-a": small + shards[:half], "part-b": shards[half:]}
PARTS = {k: v for k, v in PARTS.items() if v}
need_part = max(sum(OVERLAY[n]["size"] for n in fs) for fs in PARTS.values()) / 1e9 + 4
step("disk", cands[STAGE_ROOT] >= need_part, free_gb=cands, stage_root=STAGE_ROOT,
     parts={k: [len(v), round(sum(OVERLAY[n]["size"] for n in v) / 1e9, 2)] for k, v in PARTS.items()}, need_gb=round(need_part, 1))


In [ ]:
# ==================================================================== CELL 4 — download (verified) and upload, part by part
from kagglesdk.blobs.types.blob_api_service import ApiBlobType
from kagglehub.gcs_upload import normalize_patterns, upload_files_and_directories
from kagglehub.handle import parse_model_handle
from kagglehub.models_helpers import create_model_if_missing, create_model_instance_or_version

def fetch(name, dest_dir, tries=4):
    """stream one file from the pinned commit, hashing while downloading; retry from scratch on any error"""
    url = f"https://huggingface.co/{REPO}/resolve/{REVISION}/{name}"
    exp = OVERLAY[name]; dest = Path(dest_dir) / name
    last = None
    for k in range(tries):
        try:
            h = hashlib.sha256(); n = 0; t = time.time()
            with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "kaggle-fp8ple-upload"}), timeout=120) as r, \
                 open(dest, "wb") as f:
                while True:
                    b = r.read(16 << 20)
                    if not b:
                        break
                    f.write(b); h.update(b); n += len(b)
            if n == exp["size"] and h.hexdigest() == exp["sha256"]:
                return {"file": name, "gb": round(n / 1e9, 2), "sec": round(time.time() - t, 1), "tries": k + 1}
            last = f"mismatch size={n} sha={h.hexdigest()[:12]}"
        except Exception as e:
            last = f"{type(e).__name__}: {e}"[:300]
        dest.unlink(missing_ok=True); time.sleep(5 * (k + 1))
    raise RuntimeError(f"{name}: {last}")

for inst, files in PARTS.items():
    part_dir = Path(STAGE_ROOT) / "fp8ple" / inst
    shutil.rmtree(part_dir, ignore_errors=True); part_dir.mkdir(parents=True)
    t = time.time()
    try:
        with ThreadPoolExecutor(4) as ex:
            rows = list(ex.map(lambda n: fetch(n, part_dir), files))
    except Exception as e:
        step(f"download_{inst}", False, error=str(e))
    on_disk = sorted(p.name for p in part_dir.iterdir() if p.is_file())
    step(f"download_{inst}", on_disk == sorted(files), files=len(rows), gb=round(sum(r["gb"] for r in rows), 2),
         sec=round(time.time() - t, 1), all_sha256_verified=True)

    handle = f"{EXPECTED_USER}/{MODEL_SLUG}/transformers/{inst}"
    t = time.time()
    try:
        h = parse_model_handle(handle)
        create_model_if_missing(h.owner, h.model)                 # new models are private
        tokens = upload_files_and_directories(str(part_dir), item_type=ApiBlobType.MODEL,
                                              ignore_patterns=normalize_patterns(default=[".cache/", ".huggingface/"], additional=None))
        try:
            create_model_instance_or_version(h, tokens, LICENSE_NAME, NOTES)
            lic = LICENSE_NAME
        except Exception as e:                                    # unknown license name: keep the uploaded blobs, use the default
            print("license", LICENSE_NAME, "rejected:", str(e)[:300])
            create_model_instance_or_version(h, tokens, None, NOTES)
            lic = "default"
        step(f"upload_{inst}", True, handle=handle, sec=round(time.time() - t, 1), license=lic)
    except Exception as e:
        step(f"upload_{inst}", False, handle=handle, sec=round(time.time() - t, 1), error=f"{type(e).__name__}: {e}"[:2000])
    shutil.rmtree(part_dir, ignore_errors=True)                   # free the disk before the next part


In [ ]:
# ==================================================================== CELL 5 — summary
REPORT["total_sec"] = round(time.time() - T0, 1)
REPORT["instances"] = {k: f"{EXPECTED_USER}/{MODEL_SLUG}/transformers/{k}" for k in PARTS}
(OUT / "upload_report.json").write_text(json.dumps(REPORT, indent=2, default=str))
print("=" * 90)
print("UPLOADED:", ", ".join(REPORT["instances"].values()))
print("Kaggle now processes the files (can take a while for ~51 GB); the model is private.")
print("total", REPORT["total_sec"], "s")
